# Library pipeline metrics dashboard

Run `library_app/app.py` first to append metrics to `library_app/data/pipeline_metrics.csv`. This notebook charts data volume and quality for the latest run, and processing duration across all recorded runs.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

candidates = [
    Path("../data/pipeline_metrics.csv"),
    Path("data/pipeline_metrics.csv"),
    Path("library_app/data/pipeline_metrics.csv"),
    Path("../library_app/data/pipeline_metrics.csv"),
]
metrics_path = next((path for path in candidates if path.is_file()), None)

if metrics_path is None:
    raise FileNotFoundError(
        "Could not find pipeline_metrics.csv. Run the pipeline first."
    )

metrics = pd.read_csv(metrics_path, parse_dates=["run_timestamp"])
metrics = metrics.sort_values("run_timestamp")
print(f"Loaded {len(metrics)} metric rows from {metrics_path.resolve()}")
metrics.tail()

In [ ]:
latest_run_id = metrics.loc[metrics["run_timestamp"].idxmax(), "run_id"]
latest_run = metrics.loc[metrics["run_id"] == latest_run_id].copy()
latest_run

In [ ]:
volume = latest_run.melt(
    id_vars="dataset_name",
    value_vars=["rows_read", "rows_written"],
    var_name="measure",
    value_name="row_count",
)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(8, 5))
sns.barplot(data=volume, x="dataset_name", y="row_count", hue="measure")
plt.title("Rows read and written in the latest run")
plt.xlabel("Dataset")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:
quality = latest_run.melt(
    id_vars="dataset_name",
    value_vars=["rows_dropped", "invalid_date_rows"],
    var_name="quality_metric",
    value_name="row_count",
)

plt.figure(figsize=(8, 5))
sns.barplot(data=quality, x="dataset_name", y="row_count", hue="quality_metric")
plt.title("Rows dropped and invalid dates in the latest run")
plt.xlabel("Dataset")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.lineplot(
    data=metrics,
    x="run_timestamp",
    y="duration_seconds",
    hue="dataset_name",
    marker="o",
    estimator=None,
)
plt.title("Processing duration by dataset across runs")
plt.xlabel("Run timestamp")
plt.ylabel("Seconds")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()